# Generate loan_outcome.csv

This notebook reads `data/loan_details.csv` and `data/customer_profile.csv`, applies Stage‑3 correlation multipliers to compute per‑loan default probabilities, samples defaults, and writes `data/loan_outcome.csv`.

Defaults follow the rules in your Stage 3 spec (base rate 17% adjusted by employment, product, city tier, channel, credit band).

In [68]:
# Read inputs
loan_path = Path('../data/loan_details.csv')
cust_path = Path('../data/customer_profile.csv')
if not loan_path.exists() or not cust_path.exists():
    raise FileNotFoundError('Ensure data/loan_details.csv and data/customer_profile.csv exist (run previous notebooks)')
loans = pd.read_csv(loan_path)
cust = pd.read_csv(cust_path)
# merge for attributes used in multipliers
df = loans.merge(cust[['customer_id','employment_type','acquisition_channel','city_tier','credit_score']], on='customer_id', how='left')
BASE_RATE = 0.145
print('Loaded', len(df), 'loan rows')
print('Base default rate set to', BASE_RATE)


Loaded 12000 loan rows
Base default rate set to 0.145


In [63]:
# Read inputs
loan_path = Path('../data/loan_details.csv')
cust_path = Path('../data/customer_profile.csv')
if not loan_path.exists() or not cust_path.exists():
    raise FileNotFoundError('Ensure data/loan_details.csv and data/customer_profile.csv exist (run previous notebooks)')
loans = pd.read_csv(loan_path)
cust = pd.read_csv(cust_path)
# merge for attributes used in multipliers
df = loans.merge(cust[['customer_id','employment_type','acquisition_channel','city_tier','credit_score']], on='customer_id', how='left')
BASE_RATE = 0.1545
print('Loaded', len(df), 'loan rows')
print('Base default rate set to', BASE_RATE)

Loaded 12000 loan rows
Base default rate set to 0.1545


In [ ]:
# Define multipliers from Stage 3 spec
emp_mult = {'Gig Worker':2.40, 'Student':0.82, 'Salaried':1.70, 'Self Employed':0.78, 'Business Owner':3.80}
prod_mult = {'BNPL':1.12, 'Education Loan':1.35, 'Personal Loan':0.74, 'SME Loan':0.84}
city_mult = {'Tier 3':1.15, 'Tier 2':1.00, 'Tier 1':0.90}
channel_mult = {'Social Media':3.25, 'Referral':0.25, 'Organic':0.63, 'DSA Agent':1.20}
# credit bands multiplier as mapping by band
def credit_multiplier(score):
    try:
        s = float(score)
    except Exception:
        return 1.0
    if s < 450:
        return 1.70
    if s <= 550:
        return 1.35
    if s <= 650:
        return 1.10
    if s <= 750:
        return 0.90
    return 0.75

In [69]:
# Compute final probability per loan
probs = []
for _, r in df.iterrows():
    p = BASE_RATE
    p *= emp_mult.get(r.get('employment_type',''), 1.0)
    p *= prod_mult.get(r.get('product_type',''), 1.0)
    p *= city_mult.get(r.get('city_tier',''), 1.0)
    p *= channel_mult.get(r.get('acquisition_channel',''), 1.0)
    p *= credit_multiplier(r.get('credit_score', None))
    # cap probability to keep the portfolio within realistic bands
    p = min(p, 0.45)
    probs.append(p)
df['default_probability'] = probs
print('Computed default probabilities (sample):')
print(df['default_probability'].describe())

Computed default probabilities (sample):
count    12000.000000
mean         0.201413
std          0.139745
min          0.030506
25%          0.079127
50%          0.160240
75%          0.287856
max          0.450000
Name: default_probability, dtype: float64


In [70]:
# Sample default flag
rand = np.random.rand(len(df))
df['default_flag'] = (rand < df['default_probability']).astype(int)
# Assign default month (for defaults only): between 2 and 8
df['default_month'] = df['default_flag'].apply(lambda x: int(np.random.randint(2,9)) if x==1 else np.nan)
df['days_to_default'] = df['default_month'].apply(lambda x: int(x*30) if not pd.isna(x) else np.nan)
# Compute amounts paid/pending
# months_paid = min(default_month-1, tenure) for defaults; tenure for non-defaults
df['tenure_months'] = df['tenure_months'].astype(int)
def months_paid(row):
    if row['default_flag']==1:
        m = max(0, int(row['default_month'])-1)
        return min(m, row['tenure_months'])
    return row['tenure_months']
df['months_paid'] = df.apply(months_paid, axis=1)
df['total_amount_paid'] = (df['months_paid'] * df['emi_amount']).round(2)
# total loan value (principal) approximated as loan_amount; pending = loan_amount - principal_paid (approx)
df['total_amount_pending'] = (df['loan_amount'] - df['total_amount_paid']).clip(lower=0).round(2)
# recovery: assume recover 10-40% of pending for defaults, else 0
df['recovery_amount'] = df.apply(lambda r: round(r['total_amount_pending'] * np.random.uniform(0.1,0.4),2) if r['default_flag']==1 else 0.0, axis=1)
# closure date: if not default, orig + tenure months; if default, orig + days_to_default
df['origination_date'] = pd.to_datetime(df['origination_date'])
df['closure_date'] = df.apply(lambda r: (r['origination_date'] + pd.DateOffset(months=int(r['tenure_months']))).date().isoformat() if r['default_flag']==0 else (r['origination_date'] + pd.Timedelta(days=int(r['days_to_default']))).date().isoformat(), axis=1)
df['write_off_flag'] = df['default_flag']  # simple flag: defaulted -> potential write-off
print('Defaults sampled: ', int(df['default_flag'].sum()), 'of', len(df))
print('Observed overall default rate:', round(df['default_flag'].mean(),4))

Defaults sampled:  2460 of 12000
Observed overall default rate: 0.205


In [71]:
# Select and save outcome fields
out = df[['loan_id','customer_id','product_type','default_flag','days_to_default','default_month','total_amount_paid','total_amount_pending','recovery_amount','closure_date','write_off_flag']].copy()
Path('../data').mkdir(parents=True, exist_ok=True)
out_path = Path('../data/loan_outcome.csv')
out.to_csv(out_path, index=False)
print(f'Wrote loan outcome to: {out_path}')

# Quick segment checks
if 'employment_type' in df.columns:
    print('Default rate by employment (sample):')
    print(df.groupby('employment_type')['default_flag'].mean())
if 'product_type' in df.columns:
    print('Default rate by product (sample):')
    print(df.groupby('product_type')['default_flag'].mean())

Wrote loan outcome to: ..\data\loan_outcome.csv
Default rate by employment (sample):
employment_type
Business Owner    0.208707
Gig Worker        0.329190
Salaried          0.109857
Self Employed     0.124803
Student           0.227059
Name: default_flag, dtype: float64
Default rate by product (sample):
product_type
BNPL              0.276911
Education Loan    0.227972
Personal Loan     0.179098
SME Loan          0.188757
Name: default_flag, dtype: float64
